In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("maqeel732/wa-fn-usec-telco-customer-churn-csv")

print("Path to dataset files:", path)

c:\Users\DanielS\Work\projects\vanilla_test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 172k/172k [00:00<00:00, 382kB/s]

Extracting files...
Path to dataset files: C:\Users\DanielS\.cache\kagglehub\datasets\maqeel732\wa-fn-usec-telco-customer-churn-csv\versions\1


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. READ IN THE DATA
# ==========================================
# Load raw tabular data from a CSV file
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Separate features (X) and target label (y)
X_raw = df.drop(columns=["Churn"]).values
y_raw = df["Churn"].values

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# Preprocessing: Scale continuous features (Z-score normalization)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert NumPy arrays to PyTorch Tensors (float32 required for linear layers)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1) # Shape: (N, 1)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)


# ==========================================
# 2. MODEL DEFINITION & SETUP
# ==========================================
class ChurnMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        return self.net(x)

model = ChurnMLP(input_dim=X_train.shape[1])
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)


# ==========================================
# 3. TRAINING LOOP
# ==========================================
epochs = 100
for epoch in range(epochs):
    model.train()
    
    # Forward Pass
    y_pred = model(X_train_tensor)
    loss = criterion(y_pred, y_train_tensor)
    
    # Backward Pass & Optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


# ==========================================
# 4. SEE THE RESULTS (EVALUATION / INFERENCE)
# ==========================================
model.eval() # Set model to evaluation mode

with torch.no_grad(): # Disable gradient computation for speed & memory efficiency
    # Get continuous probability outputs (between 0.0 and 1.0)
    test_probabilities = model(X_test_tensor)
    
    # Convert probabilities to discrete binary predictions (0 or 1) using a 0.5 threshold
    test_predictions = (test_probabilities >= 0.5).float()
    
    # Calculate classification accuracy
    correct = (test_predictions == y_test_tensor).sum().item()
    accuracy = correct / len(y_test_tensor)

# Print direct sample predictions
print("--- Sample Predictions ---")
print("Raw Probabilities (First 5):", test_probabilities[:5].squeeze().tolist())
print("Predicted Classes (First 5):", test_predictions[:5].squeeze().tolist())
print("Actual Ground Truth (First 5):", y_test_tensor[:5].squeeze().tolist())
print(f"\nOverall Test Accuracy: {accuracy * 100:.2f}%")

ValueError: could not convert string to float: '4223-BKEOR'